# OctWave3 — Kaggle Image Classification (Colab T4)

**Runtime → Change runtime type → T4 GPU** before running anything.

This notebook is deliberately thin: all real logic lives in `src/*.py` in the GitHub repo.
Edit those files locally, `git push`, then re-run the **Pull repo** cell here. You should
almost never need to edit this notebook, which keeps `.ipynb` diffs tiny.

Checkpoints are written to **Google Drive**, so a disconnect costs one epoch, not the run.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Install extra packages

Torch/torchvision already ship with Colab — only these are missing.

In [ ]:
!pip install -q timm albumentations kaggle

## 3. Mount Google Drive

Checkpoints live here so training survives a dropped session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_OUT = Path('/content/drive/MyDrive/octwave3/outputs')
(DRIVE_OUT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_OUT / 'submissions').mkdir(parents=True, exist_ok=True)
print(DRIVE_OUT)

## 4. Clone / pull the repo

Set `REPO_URL` once. Re-run this cell every session, and again after every push
from your laptop — it is the only step needed to get new code into Colab.

For a private repo, create a GitHub fine-grained token and use:
`https://<TOKEN>@github.com/<user>/<repo>.git` (store the token in Colab Secrets 🔑, not here).

In [ ]:
import os
REPO_URL  = 'https://github.com/<your-username>/OctWave3.git'   # <-- EDIT
REPO_NAME = 'OctWave3'

if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone $REPO_URL /content/$REPO_NAME
else:
    !cd /content/$REPO_NAME && git pull

%cd /content/$REPO_NAME
import sys; sys.path.insert(0, '/content/' + REPO_NAME)
!git log --oneline -3

## 5. Download the competition data

Upload your `kaggle.json` (Kaggle → Account → Create New API Token) when prompted,
or store its contents in Colab Secrets. Data goes to local disk (`/content`), **not**
Drive — local disk is far faster to read from, and re-downloading each session is cheap.

In [ ]:
from google.colab import files
import os, json

if not os.path.exists('/root/.kaggle/kaggle.json'):
    files.upload()                      # pick kaggle.json
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

COMP = '<competition-slug>'             # <-- EDIT once the competition opens
!kaggle competitions download -c $COMP -p /content/data
!unzip -q -o /content/data/*.zip -d /content/data
!ls /content/data

## 6. Configure the experiment

Everything tunable is in `src/config.py`; override per-run here.

In [ ]:
from pathlib import Path
import importlib, src.config, src.dataset, src.model, src.train, src.utils
for m in (src.config, src.utils, src.dataset, src.model, src.train):
    importlib.reload(m)
from src.config import cfg

cfg.data_dir = Path('/content/data')
cfg.out_dir  = DRIVE_OUT          # checkpoints survive disconnects

cfg.exp_name    = 'exp01_baseline'
cfg.model_name  = 'tf_efficientnet_b0'
cfg.img_size    = 224
cfg.batch_size  = 64
cfg.epochs      = 10
cfg.lr          = 3e-4
cfg.num_classes = 10              # <-- set to the real class count

cfg.to_dict()

## 7. Train

Safe to re-run after a disconnect — it resumes from `*_last.pt` in Drive automatically.
Keep the browser tab open; Colab kills idle sessions.

In [ ]:
from src.train import run_fold
best = run_fold(cfg, fold=0)

## 8. Training curves

Reads the `run_log.jsonl` that `train.py` appends to on every epoch.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

log = pd.read_json(cfg.out_dir / 'run_log.jsonl', lines=True)
log = log[log.exp == cfg.exp_name]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(log.epoch, log.train_loss, label='train')
ax[0].plot(log.epoch, log.val_loss, label='valid')
ax[0].set_title('loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(log.epoch, log.acc, label='acc')
ax[1].plot(log.epoch, log.f1, label='macro f1')
ax[1].set_title('metrics'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()
log.tail()

## 9. Predict & submit

In [ ]:
import pandas as pd
from pathlib import Path
from src.dataset import build_dataframe
from src.predict import predict, make_submission

classes = build_dataframe(cfg).attrs['classes']
test_df = pd.DataFrame({'path': sorted(Path('/content/data/test').rglob('*' + cfg.image_ext))})

ckpts = sorted((cfg.out_dir / 'checkpoints').glob(f'{cfg.exp_name}_f*_best.pt'))
probs = predict(cfg, ckpts, test_df)
sub   = make_submission(cfg, probs, test_df, classes, f'{cfg.exp_name}.csv')
sub.head()

In [ ]:
!kaggle competitions submit -c $COMP \
    -f {cfg.out_dir}/submissions/{cfg.exp_name}.csv \
    -m "{cfg.exp_name}: {cfg.model_name} @ {cfg.img_size}"

## 10. After the run — record it

1. Add one row to `logs/EXPERIMENTS.md` (model, img size, LR, CV, LB).
2. Note anything notable in `logs/CHANGELOG.md`.
3. If you changed this notebook: **File → Save a copy in GitHub**, target
   `notebooks/01_train_colab.ipynb`, and tick *Include a link to Colab*.
   Run **Edit → Clear all outputs** first so the diff stays small.